In [ ]:
#######################################################################################################################################################################
########################################### This notebook takes the cleaned data file and format it for Google Earth Engine ##########################################
#######################################################################################################################################################################

In [1]:
# Import the necessary packages
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import json

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import ee # Import the Python API package
import geemap # The geemap Python package is built upon the ipyleaflet and folium packages and implements several methods for interacting with Earth Engine data layer

try:
    ee.Initialize(project='amibea') # Initialize the API
except Exception as e:
    ee.Authenticate() # Authenticate to the Earth Engine servers
    ee.Initialize(project='amibea') # Initialize the API

In [2]:
# Import the raw data from a CSV file into a data frame
pathToFile = r'E:\Amibatec\Updated2026\TA_Diversity.csv'
rawData = pd.read_csv(pathToFile,sep=";",encoding= 'unicode_escape')
Data = pd.DataFrame(rawData)
print(Data.shape)
print(Data.head(10))

(950, 3)
    latitude   longitude  Diversity
0  74.471044  -20.573984  17.341311
1  69.823128   27.171842  18.251664
2  69.466878   91.434332  26.067712
3  69.383545   91.550998  23.230366
4  69.383545   91.559332  21.488078
5  69.254378  -53.517729  27.705175
6  68.885628   21.053093  13.091230
7  68.623128 -149.580213  19.016547
8  68.364795   19.582260  14.614873
9  68.356461   18.811427  11.469787


In [3]:
# Export the result to disc
Data.to_csv(r'E:\Amibatec\Updated2026\TA_Diversity_GEEformated.csv', index = False)

In [4]:
# Export an ee.FeatureCollection as an Earth Engine asset.
# create a tmp gdf

df = pd.read_csv(r'E:\Amibatec\Updated2026\TA_Diversity_GEEformated.csv', sep=',', engine='python')
gdf = gpd.GeoDataFrame(
    df, 
    crs='EPSG:4326', 
    geometry = gpd.points_from_xy(
        df['longitude'], 
        df['latitude']
    )
)
    
# convert it into geo-json 
json_df = json.loads(gdf.to_json())
    
# create a gee object with geemap
ee_object = geemap.geojson_to_ee(json_df)
            
#create and launch the task
task = ee.batch.Export.table.toAsset(
    collection= ee_object,
    description= 'exportToTableAsset',
    assetId= 'projects/amibea/assets/TA_Diversity_GEEformated_2026',
)
task.start()